# SAM Auto-Segmentation Pipeline

Converts YOLO bounding-box labels to segmentation polygons using [Segment Anything (SAM)](https://github.com/facebookresearch/segment-anything).

## Imports & Configuration

In [7]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import glob
import traceback

import cv2
import numpy as np
import torch
from tqdm import tqdm
from segment_anything import sam_model_registry, SamPredictor

# ── Constants ─────────────────────────────────────────────────────────────────
MIN_MASK_AREA    = 10    # reject SAM masks smaller than this (pixels)
MIN_POLY_POINTS  = 3     # minimum polygon vertices required
DEBUG_OUTPUT_DIR = "readme_images"

SAM_CHECKPOINTS = {
    "vit_b": (
        "sam_vit_b_01ec64.pth",
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
    ),
    "vit_h": (
        "sam_vit_h_4b8939.pth",
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth",
    ),
}

## Utility Functions

`setup_sam` · `yolo_to_pixel_bbox` · `mask_to_yolo_polygon`

In [8]:
# ─── 1. Model Setup ───────────────────────────────────────────────────────────

def setup_sam(checkpoint_path, model_type="vit_b", device="cpu"):
    """Load a SAM model from disk and return a SamPredictor."""
    print(f"Loading SAM ({model_type}) from {checkpoint_path}...")
    try:
        sam = sam_model_registry[model_type](checkpoint=checkpoint_path)
        sam.to(device=device)
        print("SAM loaded successfully.")
        return SamPredictor(sam)
    except Exception as e:
        print(f"Failed to load SAM: {e}")
        traceback.print_exc()
        return None


# ─── 2. Geometry Utilities ────────────────────────────────────────────────────

def yolo_to_pixel_bbox(yolo_bbox, img_width, img_height):
    """
    Convert a YOLO bounding box (normalised x_c, y_c, w, h) to
    pixel coordinates (x_min, y_min, x_max, y_max).
    """
    x_c, y_c, w, h = yolo_bbox
    x_c *= img_width;  y_c *= img_height
    w   *= img_width;  h   *= img_height

    x_min = max(0,          int(x_c - w / 2))
    y_min = max(0,          int(y_c - h / 2))
    x_max = min(img_width,  int(x_c + w / 2))
    y_max = min(img_height, int(y_c + h / 2))

    return [x_min, y_min, x_max, y_max]


def mask_to_yolo_polygon(mask, img_width, img_height, simplify=True, tolerance=0.01):
    """
    Convert a binary segmentation mask to a YOLO polygon
    (flat list of normalised x, y coordinates).

    tolerance — Douglas-Peucker simplification strength:
        lower  (e.g. 0.001) → more points, finer detail
        higher (e.g. 0.05)  → fewer points, coarser shape
    """
    try:
        contours, _ = cv2.findContours(
            mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if not contours:
            print("  No contours found in mask.")
            return None

        contour = max(contours, key=cv2.contourArea)
        if len(contour) < MIN_POLY_POINTS:
            print("  Contour has too few points.")
            return None

        if simplify:
            epsilon = tolerance * cv2.arcLength(contour, True)
            contour = cv2.approxPolyDP(contour, epsilon, True)

        points = contour.reshape(-1, 2)
        return [coord
                for x, y in points
                for coord in (x / img_width, y / img_height)]

    except Exception as e:
        print(f"  mask_to_yolo_polygon error: {e}")
        traceback.print_exc()
        return None

## Pipeline Functions

`_predict_best_mask` · `_save_debug_image` · `_process_object` · `process_image` · `convert_dataset` · `main`

In [9]:
# ─── 3. SAM & Debug Helpers ───────────────────────────────────────────────────

def _predict_best_mask(predictor, box):
    """
    Run SAM on a pixel bounding box and return the highest-scoring mask.
    Returns (mask, area_in_pixels) or (None, 0) on failure.
    """
    try:
        masks, scores, _ = predictor.predict(
            box=box,
            multimask_output=True,
            point_coords=None,
            point_labels=None,
        )
        best = np.argmax(scores)
        mask = masks[best]
        return mask, int(mask.sum())
    except Exception as e:
        print(f"    SAM prediction error: {e}")
        traceback.print_exc()
        return None, 0


def _save_debug_image(debug_img, image_path):
    """Save a debug overlay image to DEBUG_OUTPUT_DIR."""
    os.makedirs(DEBUG_OUTPUT_DIR, exist_ok=True)
    stem     = os.path.splitext(os.path.basename(image_path))[0]
    out_path = os.path.join(DEBUG_OUTPUT_DIR, f"{stem}_debug.jpg")
    cv2.imwrite(out_path, debug_img)
    print(f"  Debug image saved → {out_path}")


def _process_object(line, obj_idx, predictor, img_width, img_height, debug_img, tolerance):
    """
    Run SAM on one YOLO bounding-box label line and produce a segmentation line.

    Parameters
    ----------
    line      : raw text line from the label file
    obj_idx   : 1-based index used in log messages
    debug_img : numpy image to draw overlays on, or None to skip drawing

    Returns
    -------
    (new_line, success)
        new_line — segmentation label string, or the original bbox line as fallback
        success  — True when SAM segmentation succeeded, False when falling back
    """
    parts = line.strip().split()
    if len(parts) < 5:
        print(f"  Object {obj_idx}: fewer than 5 fields — skipping.")
        return None, False

    class_id  = parts[0]
    bbox      = [float(c) for c in parts[1:5]]
    pixel_box = yolo_to_pixel_bbox(bbox, img_width, img_height)
    box       = np.array(pixel_box)

    # Reject degenerate boxes
    if box[0] >= box[2] or box[1] >= box[3]:
        print(f"  Object {obj_idx}: degenerate box {box} — bbox fallback.")
        return line.strip(), False

    # Draw bounding box on the debug image
    if debug_img is not None:
        x_min, y_min, x_max, y_max = pixel_box
        cv2.rectangle(debug_img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

    # SAM mask prediction
    print(f"  Object {obj_idx}: predicting mask...")
    mask, area = _predict_best_mask(predictor, box)

    if mask is None:
        return line.strip(), False
    print(f"  Object {obj_idx}: mask area = {area} px")

    if area < MIN_MASK_AREA:
        print(f"  Object {obj_idx}: mask too small — bbox fallback.")
        return line.strip(), False

    # Convert mask to a YOLO polygon
    polygon = mask_to_yolo_polygon(mask, img_width, img_height, tolerance=tolerance)

    if polygon is None or len(polygon) < MIN_POLY_POINTS * 2:
        print(f"  Object {obj_idx}: polygon too small — bbox fallback.")
        return line.strip(), False

    # Draw polygon on the debug image
    if debug_img is not None:
        pts = np.array([
            [int(polygon[j] * img_width), int(polygon[j + 1] * img_height)]
            for j in range(0, len(polygon), 2)
        ])
        cv2.polylines(debug_img, [pts], isClosed=True, color=(0, 0, 255), thickness=2)

    seg_line = f"{class_id} " + " ".join(f"{c:.6f}" for c in polygon)
    return seg_line, True


# ─── 4. Image & Dataset Pipeline ─────────────────────────────────────────────

def process_image(image_path, label_path, predictor,
                  output_label_path=None, save_debug_viz=False, tolerance=0.01):
    """
    Segment every labelled object in one image using SAM bounding-box prompts.

    Writes a YOLO segmentation label file and (optionally) a debug overlay image.
    Returns the output label path on success, or None on failure.
    """
    print(f"\nProcessing {os.path.basename(image_path)}...")

    # Load & validate image
    image = cv2.imread(image_path)
    if image is None:
        print("  Error: could not read image.")
        return None

    img_height, img_width = image.shape[:2]
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    print(f"  Size: {img_width}x{img_height}")

    try:
        predictor.set_image(image_rgb)
    except Exception as e:
        print(f"  Error setting SAM image: {e}")
        return None

    # Load label file
    if not os.path.exists(label_path):
        print(f"  Error: label file not found: {label_path}")
        return None

    with open(label_path) as f:
        lines = f.readlines()
    print(f"  Found {len(lines)} object(s).")

    # Process each labelled object
    debug_img = image.copy() if save_debug_viz else None
    new_lines = []
    successes = 0
    fallbacks = 0

    for i, line in enumerate(lines, start=1):
        try:
            new_line, success = _process_object(
                line, i, predictor, img_width, img_height, debug_img, tolerance
            )
            if new_line is not None:
                new_lines.append(new_line)
                if success:
                    successes += 1
                else:
                    fallbacks += 1
        except Exception as e:
            print(f"  Unexpected error on object {i}: {e}")
            traceback.print_exc()
            if line.strip():
                new_lines.append(line.strip())
                fallbacks += 1

    print(f"  Result: {successes} segmented, {fallbacks} bbox fallback(s).")

    # Write output label file
    if output_label_path is None:
        output_label_path = label_path.replace(".txt", "_seg.txt")

    with open(output_label_path, "w") as f:
        f.write("\n".join(new_lines) + "\n")

    # Save debug overlay
    if save_debug_viz and (successes + fallbacks) > 0:
        _save_debug_image(debug_img, image_path)

    return output_label_path if (successes + fallbacks) > 0 else None


def convert_dataset(
    images_dir,
    labels_dir,
    output_labels_dir=None,
    sam_checkpoint="sam_vit_b_01ec64.pth",
    model_type="vit_b",
    device="cpu",
    debug_first_n=4,
    tolerance=0.01,
):
    """
    Run SAM segmentation on every labelled image in a YOLO dataset folder.

    Parameters
    ----------
    images_dir        : folder of source images
    labels_dir        : folder of YOLO bounding-box .txt files
    output_labels_dir : destination for segmentation labels (default: labels_dir + '_seg')
    sam_checkpoint    : path to SAM .pth weights file
    model_type        : 'vit_b' (fast) or 'vit_h' (accurate)
    device            : 'cuda' or 'cpu'
    debug_first_n     : save a debug overlay image for the first N images
    tolerance         : Douglas-Peucker polygon simplification (0.001–0.05)
    """
    if output_labels_dir is None:
        output_labels_dir = labels_dir + "_seg"
    os.makedirs(output_labels_dir, exist_ok=True)

    predictor = setup_sam(sam_checkpoint, model_type, device)
    if predictor is None:
        return None

    # Collect images (capped at 5; remove the slice to process all)
    image_paths = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
        image_paths.extend(glob.glob(os.path.join(images_dir, ext)))
    image_paths = image_paths[:5]

    print(f"Images found : {len(image_paths)}")
    print(f"Tolerance    : {tolerance}")
    print(f"Debug images : first {debug_first_n}\n")

    processed = successful = 0

    for i, image_path in enumerate(tqdm(image_paths, desc="Segmenting")):
        base        = os.path.splitext(os.path.basename(image_path))[0]
        label_path  = os.path.join(labels_dir,        base + ".txt")
        output_path = os.path.join(output_labels_dir, base + ".txt")

        if not os.path.exists(label_path):
            continue

        processed += 1
        result = process_image(
            image_path, label_path, predictor,
            output_label_path=output_path,
            save_debug_viz=(i < debug_first_n),
            tolerance=tolerance,
        )
        if result:
            successful += 1

    print(f"\nComplete — {successful}/{processed} images segmented.")
    print(f"Labels saved to: {output_labels_dir}")
    return output_labels_dir


# ─── 5. Entry Point ───────────────────────────────────────────────────────────

def main(
    images_dir,
    labels_dir,
    output_dir,
    model_type="vit_b",
    debug_first_n=4,
    tolerance=0.01,
):
    """
    Top-level entry point: download SAM weights if needed, then run the pipeline.

    Parameters
    ----------
    images_dir    : folder of source images
    labels_dir    : folder of YOLO bounding-box .txt labels
    output_dir    : destination for segmentation label files
    model_type    : 'vit_b' (faster) or 'vit_h' (more accurate)
    debug_first_n : save debug overlay images for the first N images
    tolerance     : polygon simplification strength
                    lower  (e.g. 0.001) → more points, finer detail
                    higher (e.g. 0.05)  → fewer points, coarser shape
                    default: 0.01
    """
    print("=== YOLOv8 Bounding Box → Segmentation Converter ===\n")

    model_path, download_url = SAM_CHECKPOINTS.get(model_type, SAM_CHECKPOINTS["vit_b"])

    if not os.path.exists(model_path):
        print(f"Downloading SAM weights ({model_type})...")
        !wget {download_url}

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Model : {model_type}  |  Device : {device}  |  Tolerance : {tolerance}\n")

    result = convert_dataset(
        images_dir=images_dir,
        labels_dir=labels_dir,
        output_labels_dir=output_dir,
        sam_checkpoint=model_path,
        model_type=model_type,
        device=device,
        debug_first_n=debug_first_n,
        tolerance=tolerance,
    )

    if result:
        print(f"\nDone! Segmentation labels saved to: {result}")
        print("\nTo train a YOLOv8 segmentation model:")
        print("  !yolo segment train data=dataset.yaml model=yolov8s-seg.pt epochs=50")
    else:
        print("\nConversion failed — check the error messages above.")

## ▶ Run

Adjust the paths and parameters below, then run this cell.

In [10]:
main(
    images_dir="other_test_img/images",
    labels_dir="other_test_img/labels",
    output_dir="other_test_img/labels_seg",
    model_type="vit_b",
    debug_first_n=4,   # save debug overlays for all 4 images
    tolerance=0.005,    # polygon detail: lower = more points, higher = fewer points
)

=== YOLOv8 Bounding Box → Segmentation Converter ===

Model : vit_b  |  Device : cpu  |  Tolerance : 0.005

Loading SAM (vit_b) from sam_vit_b_01ec64.pth...
SAM loaded successfully.
Images found : 4
Tolerance    : 0.005
Debug images : first 4



Segmenting:   0%|          | 0/4 [00:00<?, ?it/s]


Processing image_1.jpeg...
  Size: 768x1024


Segmenting:  25%|██▌       | 1/4 [00:04<00:13,  4.34s/it]

  Found 1 object(s).
  Object 1: predicting mask...
  Object 1: mask area = 121068 px
  Result: 1 segmented, 0 bbox fallback(s).
  Debug image saved → readme_images/image_1_debug.jpg

Processing image_2.jpeg...
  Size: 768x1024


...

  Found 1 object(s).
  Object 1: predicting mask...
  Object 1: mask area = 46487 px
  Result: 1 segmented, 0 bbox fallback(s).
  Debug image saved → readme_images/image_2_debug.jpg

Processing image_3.jpeg...
  Size: 768x1024


Segmenting:  75%|███████▌  | 3/4 [00:11<00:03,  3.78s/it]

  Found 1 object(s).
  Object 1: predicting mask...
  Object 1: mask area = 139373 px
  Result: 1 segmented, 0 bbox fallback(s).
  Debug image saved → readme_images/image_3_debug.jpg

Processing image_4.jpeg...
  Size: 768x1024


Segmenting: 100%|██████████| 4/4 [00:15<00:00,  3.79s/it]

  Found 1 object(s).
  Object 1: predicting mask...
  Object 1: mask area = 98359 px
  Result: 1 segmented, 0 bbox fallback(s).
  Debug image saved → readme_images/image_4_debug.jpg

Complete — 4/4 images segmented.
Labels saved to: other_test_img/labels_seg

Done! Segmentation labels saved to: other_test_img/labels_seg

To train a YOLOv8 segmentation model:
  !yolo segment train data=dataset.yaml model=yolov8s-seg.pt epochs=50
